In [9]:
import sys
sys.path.append("../src")
from embedding import HuggingFaceEmbeddingsLC
from qdrant_client import QdrantClient
from inferencer import GeminiInferencer

embedder = HuggingFaceEmbeddingsLC(show_progress=False)
qdrant_client = QdrantClient("http://localhost:6333")
inferencer = GeminiInferencer()

MPS is available. Using Apple Silicon GPU for embeddings.


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 63277.88it/s]


In [2]:
query = "¿Qué se considera oficialmente un \"animal de compañía\"?"

1. Embed the query

In [3]:
query_embedding = embedder.embed_query(query)
query_embedding

[-0.05872158706188202,
 -0.0030163764022290707,
 -0.032460276037454605,
 -0.002988502848893404,
 -0.012316626496613026,
 -0.05690024048089981,
 -0.0062971534207463264,
 -0.015213200822472572,
 0.008763411082327366,
 -0.01986546255648136,
 0.019346654415130615,
 0.014072242192924023,
 -0.010613399557769299,
 0.0007946271798573434,
 0.011395891197025776,
 -0.008094600401818752,
 0.004396609030663967,
 -0.045769963413476944,
 0.013428946025669575,
 -0.04487159103155136,
 -0.04308000206947327,
 -0.028201507404446602,
 0.03816661611199379,
 0.016430627554655075,
 0.03322987258434296,
 0.024101199582219124,
 0.006196015048772097,
 -0.0063247280195355415,
 -0.00852308701723814,
 0.01855645701289177,
 0.000205797899980098,
 -0.04455972835421562,
 -0.046718843281269073,
 -0.01835298351943493,
 -0.023280005902051926,
 0.0013808633666485548,
 0.004495405126363039,
 -0.020731760188937187,
 -0.03742293268442154,
 -0.004694605246186256,
 -0.033747874200344086,
 -0.02535560168325901,
 0.0061337989754

2. Retrieval

In [6]:
results = qdrant_client.query_points(
    collection_name="documents",
    query=query_embedding,
    limit=5,
    with_payload=True # to get also the text
)
for result in results.points:
    print(f"- Score: {result.score:.2f}, File {result.payload['source_file']} (Page: {result.payload['page']}): {result.payload['text'][:100]}...")

results.points[0].payload["text"]
results

- Score: 0.69, File BOE-A-2023-7936.pdf (Page: 49): ía en el supuesto de que, perdiendo su fin 
productivo, el propietario decidiera inscribirlo como an...
- Score: 0.68, File BOE-A-2023-7936.pdf (Page: 6): etológicas, pueda adaptarse a la 
cautividad y que su tenencia no tenga como destino su consumo o el...
- Score: 0.65, File BOE-A-2023-7936.pdf (Page: 25): e residencia de animales, la distribución de animales entre el alumnado y 
cualquier otra práctica s...
- Score: 0.64, File BOE-A-2023-7936.pdf (Page: 6): especies y poblaciones de fauna cuyo geno/fenotipo no se ha visto afectado por la 
selección humana,...
- Score: 0.64, File BOE-A-2023-7936.pdf (Page: 50): industrial o cualquier otro fin comercial o lucrativo y que, en el caso de los 
animales silvestres,...


QueryResponse(points=[ScoredPoint(id='e37892f9-e03f-48a9-bb94-ad3d1a96e2fe', version=526, score=0.6886617, payload={'producer': 'Antenna House PDF Output Library 6.6.1477 (Linux64)', 'creator': 'eBOE', 'creationdate': '2023-03-28T22:54:35+01:00', 'keywords': 'LEY 7/2023 de 28/03/2023;JEFATURA DEL ESTADO;BOE-A-2023-7936;BOE 75 de 2023;7936;29/03/2023', 'moddate': '2023-03-28T23:03:26+02:00', 'trapped': '/False', 'subject': 'BOE-A-2023-7936', 'author': 'JEFATURA DEL ESTADO', 'title': 'Disposición 7936 del BOE núm. 75 de 2023', 'source': '/Users/aingeru/workspace/Máster - TFM/data/raw/BOE-A-2023-7936.pdf', 'total_pages': 54, 'page': 49, 'page_label': '50', 'source_file': 'BOE-A-2023-7936.pdf', 'text': 'ía en el supuesto de que, perdiendo su fin \nproductivo, el propietario decidiera inscribirlo como animal de compañía en el \nRegistro de Animales de compañía.\n3.\u2003Animal de compañía: animal doméstico o silvestre en cautividad \nmantenido por el ser humano, principalmente en el hogar,

3. Generation

In [ ]:
contexto = "\n\n".join([result.payload["text"] for result in results.points])

prompt = f"""Responde a la pregunta basándote únicamente en el contexto proporcionado.
Contexto: {contexto}

Pregunta: {query}"""

prompt

'Responde a la pregunta basándote únicamente en el contexto proporcionado.\nContexto: ía en el supuesto de que, perdiendo su fin \nproductivo, el propietario decidiera inscribirlo como animal de compañía en el \nRegistro de Animales de compañía.\n3.\u2003Animal de compañía: animal doméstico o silvestre en cautividad \nmantenido por el ser humano, principalmente en el hogar, siempre que se pueda \ntener en buenas condiciones de bienestar que respeten sus necesidades \netológicas, pueda adaptarse a la cautividad y que su tenencia no tenga como \ndestino su consumo o el aprovechamiento de sus producciones o cualquier uso \nBOLETÍN OFICIAL DEL ESTADO\nNúm. 75 Miércoles 29 de marzo de 2023 Sec. I.   Pág. 45667\ncve: BOE-A-2023-7936\nVerificable en https://www.boe.es\n\netológicas, pueda adaptarse a la \ncautividad y que su tenencia no tenga como destino su consumo o el aprovechamiento \nde sus producciones o cualquier uso industrial o cualquier otro fin comercial o lucrativo y \nque, en el 

In [10]:
response_text = inferencer.infer(prompt)
print("Respuesta:", response_text)

Respuesta: Según el contexto proporcionado, se considera animal de compañía a todo animal doméstico o silvestre en cautividad mantenido por el ser humano, principalmente en el hogar, siempre que cumpla los siguientes requisitos:

1. Que se pueda tener en buenas condiciones de bienestar que respeten sus necesidades etológicas.
2. Que pueda adaptarse a la cautividad.
3. Que su tenencia no tenga como destino su consumo, el aprovechamiento de sus producciones, ni ningún fin industrial, comercial o lucrativo.
4. En el caso de los animales silvestres, que su especie esté incluida en el listado positivo de animales de compañía.

Adicionalmente, el texto establece las siguientes precisiones:
* **Perros, gatos y hurones:** Serán considerados animales de compañía en todo caso, independientemente del fin al que se destinen, el lugar en el que habiten o de donde procedan.
* **Animales de producción:** Solo se considerarán animales de compañía en el supuesto de que, perdiendo su fin productivo, el 